# Multi-class topic classifier v5

6-class routing on **real data**: `domain` + 5 news topics.

**Data:** `data/real_multiclass_v5.csv` (`doc_id, text, topic`).

**Split:** grouped holdout by `doc_id` (no chunk leakage). Layer B abstain deferred.

Binary baseline: [`ml_binary_domain_v5.ipynb`](ml_binary_domain_v5.ipynb).

## 1. Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import label_binarize
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    accuracy_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


In [ ]:
def resolve_data_path(filename: str = "real_multiclass_v5.csv") -> Path:
    candidates = [
        Path.cwd() / "data" / filename,
        Path.cwd() / "../data" / filename,
        Path.cwd().parent / "data" / filename,
    ]
    for path in candidates:
        if path.resolve().exists():
            return path.resolve()
    raise FileNotFoundError(
        f"Could not find {filename}. Tried:\n" + "\n".join(str(p.resolve()) for p in candidates)
    )

DATA_PATH = resolve_data_path()
print("Loading:", DATA_PATH)


## 2. Load

In [ ]:
df = pd.read_csv(DATA_PATH)
print("shape:", df.shape)
print("columns:", list(df.columns))
print("nulls:", df[["doc_id", "text", "topic"]].isna().sum().to_dict())
print("\n# doc_id groups:", df["doc_id"].nunique())
print("\ntopic balance:")
print(df["topic"].value_counts().sort_index())


## 3. Grouped train / test split

Hold out entire `doc_id` groups so chunks from the same document never appear in both train and test.

In [ ]:
def grouped_train_test_split(df, test_size=0.2, random_state=42):
    """Hold out whole doc_id groups; stratify on topic at doc level."""
    doc_meta = df.groupby("doc_id", as_index=False).agg(topic=("topic", "first"))
    train_docs, test_docs = train_test_split(
        doc_meta["doc_id"],
        test_size=test_size,
        stratify=doc_meta["topic"],
        random_state=random_state,
    )
    test_set = set(test_docs)
    train_df = df.loc[~df["doc_id"].isin(test_set)].copy()
    test_df = df.loc[df["doc_id"].isin(test_set)].copy()
    return train_df, test_df

train_df, test_df = grouped_train_test_split(df, test_size=0.2, random_state=RANDOM_STATE)
X_train, y_train = train_df["text"], train_df["topic"]
X_test, y_test = test_df["text"], test_df["topic"]

print(f"train rows: {len(train_df):,}  test rows: {len(test_df):,}")
print(f"train docs: {train_df['doc_id'].nunique():,}  test docs: {test_df['doc_id'].nunique():,}")
print("\ntrain topic counts:")
print(y_train.value_counts().sort_index())
print("\ntest topic counts:")
print(y_test.value_counts().sort_index())


## 4. TF-IDF + LR grid search (F1-macro)

In [ ]:
pipe = Pipeline([
    ("tfidf", TfidfVectorizer(sublinear_tf=True)),
    ("clf", LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        solver="lbfgs",
        random_state=RANDOM_STATE,
    )),
])

param_grid = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__min_df": [1, 2],
    "tfidf__max_features": [5000, 10000, None],
    "clf__C": [0.1, 1.0, 10.0],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

search = GridSearchCV(
    pipe,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    refit=True,
    verbose=1,
)

search.fit(X_train, y_train)
best_model = search.best_estimator_
CLASS_NAMES = list(best_model.named_steps["clf"].classes_)

print("best CV F1-macro:", round(search.best_score_, 4))
print("best params:")
for k, v in search.best_params_.items():
    print(f"  {k}: {v}")
print("classes:", CLASS_NAMES)


## 5. Holdout metrics

In [ ]:
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)

print("=== Holdout (grouped by doc_id) ===")
print("accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("f1_macro:", round(f1_score(y_test, y_pred, average="macro"), 4))
print("\nclassification report:")
print(classification_report(y_test, y_pred, digits=3))

if "domain" in CLASS_NAMES:
    print("\n--- domain class ---")
    print("precision:", round(precision_score(y_test, y_pred, labels=["domain"], average="macro"), 4))
    print("recall:", round(recall_score(y_test, y_pred, labels=["domain"], average="macro"), 4))

fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax, cmap="Blues")
ax.set_title("Holdout confusion matrix (v5)")
plt.tight_layout()
plt.show()

# macro OvR ROC-AUC
y_bin = label_binarize(y_test, classes=CLASS_NAMES)
try:
    auc = roc_auc_score(y_bin, y_proba, average="macro", multi_class="ovr")
    print("\nmacro OvR ROC-AUC:", round(auc, 4))
except ValueError as e:
    print("ROC-AUC skipped:", e)


## 6. Explainability (signed LR coef)

In [ ]:
TOP_N = 12
tfidf = best_model.named_steps["tfidf"]
clf = best_model.named_steps["clf"]
feature_names = np.array(tfidf.get_feature_names_out())
coefs = clf.coef_

def plot_class_coefs(ax, cls: str, row: np.ndarray, names: np.ndarray, top_n: int = TOP_N) -> None:
    pos_idx = np.argsort(row)[-top_n:][::-1]
    neg_idx = np.argsort(row)[:top_n]
    idx = np.concatenate([pos_idx, neg_idx])
    vals = row[idx]
    labels = names[idx]
    colors = ["#2ca02c" if v >= 0 else "#d62728" for v in vals]
    y_pos = np.arange(len(idx))
    ax.barh(y_pos, vals, color=colors)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(labels, fontsize=8)
    ax.invert_yaxis()
    ax.set_title(cls, fontsize=10)
    ax.axvline(0, color="k", lw=0.5)

ncols = 3
nrows = int(np.ceil(len(CLASS_NAMES) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(14, 3.5 * nrows))
axes = np.atleast_1d(axes).ravel()
for ax, cls, row in zip(axes, CLASS_NAMES, coefs):
    plot_class_coefs(ax, cls, row, feature_names)
for ax in axes[len(CLASS_NAMES):]:
    ax.axis("off")
fig.suptitle(f"Top {TOP_N} LR coef per class (+ toward, − away)", y=1.02)
plt.tight_layout()
plt.show()


## 7. Save model

In [ ]:
import joblib

def resolve_models_dir() -> Path:
    for base in [Path.cwd(), Path.cwd().parent]:
        d = base / "data" / "models"
        if d.parent.joinpath("real_multiclass_v5.csv").exists() or (base / "data").exists():
            d.mkdir(parents=True, exist_ok=True)
            return d.resolve()
    raise FileNotFoundError("Could not resolve data/models directory")

MODELS_DIR = resolve_models_dir()
MODEL_PATH = MODELS_DIR / "ml_multi_v5.joblib"
joblib.dump(best_model, MODEL_PATH)
print("saved:", MODEL_PATH)
